# Assignment 3: EDA for the report (Dataset and preprocessing)
Reads every GT PNG of **train, val and test** and prints the two tables of the report:

| Table | Columns |
| :--- | :--- |
| 1. Splits | Split, Pairs, Source, Valid GT pixels, GT density |
| 2. GT depth distribution | GT depth range, pixels and share for train / val / test |

Valid GT pixel = GT > 0. Ranges match the metric: [0, 20), [20, 40), [40, 80], above 80 m.
Run the cells top to bottom (CPU runtime is enough).

In [ ]:
# @title 1. Settings + mount Drive
DATA_DIR = "/content/kitti"                                     # dataset on the fast Colab disk
DRIVE_TARS = "/content/drive/MyDrive/KITTI_DC/dataset"          # train.tar / val.tar / test.tar
OUT_DIR = "/content/drive/MyDrive/KITTI_DC/eda"                 # csv tables are saved here
SPLITS = ["train", "val", "test"]

import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# @title 2. Load the dataset to the Colab disk (skipped if already there)
import subprocess

if not all(os.path.isdir(f"{DATA_DIR}/{s}") for s in SPLITS):
    os.makedirs(DATA_DIR, exist_ok=True)
    for s in SPLITS:
        print("extracting", s)
        subprocess.run(["tar", "-xf", f"{DRIVE_TARS}/{s}.tar", "-C", DATA_DIR], check=True)

for s in SPLITS:
    print(s, len(os.listdir(f"{DATA_DIR}/{s}/rgb")), "pairs")

In [ ]:
# @title 3. Count GT pixels per image
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np
from PIL import Image

RANGES = ["0-20 m", "20-40 m", "40-80 m", "above 80 m"]

def count_gt(path):
    """one GT png -> [all pixels, valid pixels, pixels in each of the 4 ranges]"""
    gt = np.asarray(Image.open(path), dtype=np.float32) / 256.0    # 16-bit png -> metres, 0 = no GT
    return np.array([
        gt.size,
        (gt > 0).sum(),
        ((gt > 0) & (gt < 20)).sum(),
        ((gt >= 20) & (gt < 40)).sum(),
        ((gt >= 40) & (gt <= 80)).sum(),
        (gt > 80).sum(),
    ], dtype=np.int64)

stats = {}    # split -> {"pairs": n, "counts": summed counts}
for s in SPLITS:
    split_dir = Path(DATA_DIR) / s
    names = sorted(p.name for p in (split_dir / "rgb").glob("*.png"))
    missing = [n for n in names if not (split_dir / "gt" / n).exists() or not (split_dir / "lidar" / n).exists()]
    assert not missing, f"{s}: {len(missing)} pairs without gt or lidar, e.g. {missing[:3]}"

    with ThreadPoolExecutor(os.cpu_count()) as ex:
        counts = sum(ex.map(count_gt, [split_dir / "gt" / n for n in names]))
    stats[s] = {"pairs": len(names), "counts": counts}
    print(f"{s}: {len(names)} pairs, {counts[1]:,} valid GT pixels")

In [ ]:
# @title 4. Table 1: splits
import pandas as pd

SOURCE = {"train": "KITTI DC train drives", "val": "held-out KITTI DC drives", "test": "val_selection_cropped"}
NAME = {"train": "Train", "val": "Validation", "test": "Test"}

rows = []
for s in SPLITS:
    c = stats[s]["counts"]
    rows.append({
        "Split": NAME[s],
        "Pairs": stats[s]["pairs"],
        "Source": SOURCE[s],
        "Valid GT pixels": int(c[1]),
        "GT density": c[1] / c[0],    # valid GT pixels / (images x H x W)
    })
table1 = pd.DataFrame(rows)
table1.to_csv(f"{OUT_DIR}/table1_splits.csv", index=False)
table1.style.format({"Pairs": "{:,}", "Valid GT pixels": "{:,}", "GT density": "{:.1%}"}).hide(axis="index")

In [ ]:
# @title 5. Table 2: GT depth distribution per split
rows = []
for i, r in enumerate(RANGES):
    row = {"GT depth range": r}
    for s in SPLITS:
        c = stats[s]["counts"]
        row[f"{NAME[s]} pixels"] = int(c[2 + i])
        row[f"{NAME[s]} share"] = c[2 + i] / c[1]    # share of the valid GT pixels of that split
    rows.append(row)
table2 = pd.DataFrame(rows)
table2.to_csv(f"{OUT_DIR}/table2_gt_depth.csv", index=False)

fmt = {c: ("{:,}" if c.endswith("pixels") else "{:.1%}") for c in table2.columns if c != "GT depth range"}
table2.style.format(fmt).hide(axis="index")

In [ ]:
# @title 6. Check + where the csv files are
for s in SPLITS:
    c = stats[s]["counts"]
    assert c[2:].sum() == c[1], f"{s}: ranges do not add up to the valid pixels"
print("ranges add up to the valid GT pixels for train, val and test")
print("saved:", f"{OUT_DIR}/table1_splits.csv", "and", f"{OUT_DIR}/table2_gt_depth.csv")